## A1 - Data acquisition and cleaning 

In [1]:
import json
import numpy as np
import pandas as pd

with open("business-licences.geojson", "r", encoding="utf-8") as file:
    geojson_data = json.load(file)

# Extract the properties from every GeoJSON feature
df = pd.json_normalize([
    feature["properties"]
    for feature in geojson_data["features"]
])

# Extract longitude and latitude from the geometry
coordinates = [
    feature["geometry"]["coordinates"]
    if feature.get("geometry") is not None
    else [np.nan, np.nan]
    for feature in geojson_data["features"]
]

df["longitude"] = [point[0] for point in coordinates]
df["latitude"] = [point[1] for point in coordinates]

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (204160, 29)


,folderyear,licencersn,licencenumber,licencerevisionnumber,businessname,businesstradename,status,issueddate,expireddate,businesstype,...,localarea,numberofemployees,feepaid,extractdate,geom,geo_point_2d,geo_point_2d.lon,geo_point_2d.lat,longitude,latitude
0,24,4518841,24-138560,10,Nobl Collective Ltd,NaN,Issued,2023-12-28T13:34:28-08:00,2024-12-31,Consulting and Management Services,...,Kitsilano,1.0,NaN,2026-07-01T02:32:17-07:00,NaN,NaN,NaN,NaN,NaN,NaN
1,24,4518843,24-138562,10,645064 BC Ltd,Trade Exchange Canada,Issued,2023-11-20T13:40:18-08:00,2024-12-31,Business Support Services,...,Kitsilano,1.0,NaN,2026-07-01T02:32:17-07:00,NaN,NaN,NaN,NaN,NaN,NaN
2,24,4518844,24-138563,10,(Louise Turgeon),Turgeon Business Consulting,Issued,2023-11-21T14:30:07-08:00,2024-12-31,Consulting and Management Services,...,Downtown,1.0,NaN,2026-07-01T02:32:17-07:00,NaN,NaN,NaN,NaN,NaN,NaN
3,24,4518848,24-138567,10,Baron Global Financial Canada Ltd,NaN,Issued,2023-12-11T10:12:12-08:00,2024-12-31,Consulting and Management Services,...,Downtown,5.0,NaN,2026-07-01T02:32:17-07:00,NaN,NaN,-123.118193,49.287734,-123.118193,49.287734
4,24,4518856,24-138576,10,Jennifer D S Dezell (Jennifer Dezell),Dentons Canada LLP,Issued,2023-12-08T16:34:17-08:00,2024-12-31,Legal Services,...,Downtown,217.0,NaN,2026-07-01T02:32:17-07:00,NaN,NaN,-123.113252,49.286653,-123.113252,49.286653


In [6]:
#declare important columns to check for missing values
important_columns = [
    "longitude",
    "latitude",
    "numberofemployees",
    "feepaid",
    "businesssubtype",
    "postalcode",
    "localarea",
    "issueddate",
    "expireddate"
]

missing_summary = pd.DataFrame({
    "missing_count": df[important_columns].isna().sum(),
    "missing_percent": (
        df[important_columns].isna().mean() * 100
    ).round(2)
})

# Group less common business types into "Other"
type_counts = df["businesstype"].value_counts()

# Calculate the running percentage of records covered
cumulative_percent = type_counts.cumsum() / type_counts.sum() * 100

# Keep categories until they cover about 85% of all records
top_types = cumulative_percent[cumulative_percent <= 85].index

# Group every remaining category as Other
df["business_type_grouped"] = df["businesstype"].where(
    df["businesstype"].isin(top_types),
    "Other"
)

display(missing_summary)

print("\nLicence statuses:")
display(df["status"].value_counts())

print("\nMost common business types:")
display(df["business_type_grouped"].value_counts())

,missing_count,missing_percent
longitude,101573,49.75
latitude,101573,49.75
numberofemployees,0,0.00
feepaid,75449,36.96
businesssubtype,182631,89.45
postalcode,95169,46.61
localarea,2714,1.33
issueddate,28823,14.12
expireddate,28799,14.11



Licence statuses:


status
Issued                  167044
Pending                  14941
Gone Out of Business     10434
Inactive                  6567
Cancelled                 5174
Name: count, dtype: int64


Most common business types:


business_type_grouped
Long-term Rental                               45415
Other                                          30906
Health Care Professionals and Services         18746
General Contractor                             15895
Short-term Rental Operator                     13411
Retail Dealer                                   9448
Consulting and Management Services              7236
Trade Contractor                                6661
Legal Services                                  6418
Restaurant                                      6207
Beauty Services                                 5426
Business Support Services                       5104
Limited Service Food Establishment              4808
Financial Services                              3695
Information Communication Technology            3588
Real Estate Services                            3342
Retail Dealer - Food                            2590
Wholesale Dealer - Non-Food                     2542
Association or Society  

In [7]:
issued_df = df[df["status"] == "Issued"].copy()

print("Original records:", len(df))
print("Issued records:", len(issued_df))
print(
    "Percentage retained:",
    round(len(issued_df) / len(df) * 100, 2),
    "%"
)

issued_missing = pd.DataFrame({
    "missing_count": issued_df[important_columns].isna().sum(),
    "missing_percent": (
        issued_df[important_columns].isna().mean() * 100
    ).round(2)
})

display(issued_missing)

Original records: 204160
Issued records: 167044
Percentage retained: 81.82 %


,missing_count,missing_percent
longitude,81154,48.58
latitude,81154,48.58
numberofemployees,0,0.00
feepaid,48229,28.87
businesssubtype,149621,89.57
postalcode,76343,45.70
localarea,2371,1.42
issueddate,155,0.09
expireddate,156,0.09


In [ ]:

display(
    issued_df[["numberofemployees", "feepaid"]]
    .describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99])
)

print(
    "Businesses with zero employees:",
    (issued_df["numberofemployees"] == 0).sum()
)

print(
    "Businesses with negative employees:",
    (issued_df["numberofemployees"] < 0).sum()
)

print(
    "Businesses with negative fees:",
    (issued_df["feepaid"] < 0).sum()
)